### RAG PIPELINE ! DATA INGESTION TO VECTOR DB PIPELINE 

In [1]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Leer todos los PDFs dentro de directory 
from pathlib import Path

def process_all_pdfs(pdf_directory): #funcion que recibe la ruta y procesa los pdfs
    all_documents = [] #stores todo los documentos en una lista 
    pdf_dir = Path(pdf_directory) #Convierte el str en un Path para usar metodos .glob()

    ## Buscar todos los PDFs de manera recursiva 

    pdf_files = list(pdf_dir.glob("**/*.pdf")) #busca los pdfs de forma recursiva






    

    print(f"Encontramos {len(pdf_files)} PDFs para procesar")

    for pdf_file in pdf_files: #por cada pdf encontrado en files
        print(f"\nProcesando... {pdf_file.name}") #
        try: 
            loader = PyMuPDFLoader(str(pdf_file)) #crea un loader para el pdf
            documents = loader.load() #lo carga y lo lleva a Document

            ## se agrega la fuente de la informacion a la metadata

            for doc in documents: 
                doc.metadata['source_file'] = pdf_file.name ## nos dice de donde viene la fuente
                doc.metadata['file_type'] = 'pdf' ## el tipo de fuente que es (siempre pdf)

            all_documents.extend(documents) #agrega los documentos a la lista original
            print(f"Se cargaron {len(documents)} paginas")

        except Exception as e:
            print(f"Bruh, tuviste un error {e}")

    print(f"\nTotal de documentos cargados: {len(all_documents)}")
    return all_documents

    ## procesa todos los PDFs en el data directory

all_pdf_documents = process_all_pdfs("../data") #ejecuta la funcion 



Encontramos 3 PDFs para procesar

Procesando... climate_change.pdf
Se cargaron 1 paginas

Procesando... importance_of_sleep.pdf
Se cargaron 1 paginas

Procesando... nutrition_basics.pdf
Se cargaron 1 paginas

Total de documentos cargados: 3


In [3]:
all_pdf_documents #La lista de documentos

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-16T23:14:11+00:00', 'source': '..\\data\\pdf\\climate_change.pdf', 'file_path': '..\\data\\pdf\\climate_change.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-16T23:14:11+00:00', 'trapped': '', 'modDate': "D:20260516231411+00'00'", 'creationDate': "D:20260516231411+00'00'", 'page': 0, 'source_file': 'climate_change.pdf', 'file_type': 'pdf'}, page_content='Climate Change\nClimate change refers to long-term shifts in global temperatures and weather patterns. While natural\nfactors have always influenced the climate, human activities since the industrial revolution have\nbecome the main driver of change.\nThe burning of fossil fuels releases carbon dioxide and other greenhouse gases into the atmosphere,\ntrapping heat and causing global temperatures to rise.

In [4]:
## Splitting del texto para obtener los chunks 

def split_documents(documents, chunk_size=1000,chunk_overlap=200):
    """Se hace split (separacion) del documento en chunks mas pequeños para mejorar el rendimiento del RAG"""

    text_splitter = RecursiveCharacterTextSplitter( #dividira nuestro texto de forma recursiva
        chunk_size=chunk_size, #tamaño max de cada chunk para caracteres 
        chunk_overlap=chunk_overlap, #cuantos caracteres se repiten entre chunk
        length_function = len, #mide el tañano del texto
        separators=["\n\n", "\n", " ", "", ","] #orden de prioridad para dividir cada chunk
    )

    split_docs = text_splitter.split_documents(documents) #aplica el splitter en todos los docs, y nos retorna una lista de chunks
    print(f"Se separaron {len(documents)} documentos entre {len(split_docs)} chunks")

    ## Nos mostrara un ejemplo del chunk

    if split_docs:
        print(f"\nEjemplo de chunking")
        print(f"Contenido: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Se separaron 3 documentos entre 3 chunks

Ejemplo de chunking
Contenido: Climate Change
Climate change refers to long-term shifts in global temperatures and weather patterns. While natural
factors have always influenced the climate, human activities since the industrial re
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-16T23:14:11+00:00', 'source': '..\\data\\pdf\\climate_change.pdf', 'file_path': '..\\data\\pdf\\climate_change.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-16T23:14:11+00:00', 'trapped': '', 'modDate': "D:20260516231411+00'00'", 'creationDate': "D:20260516231411+00'00'", 'page': 0, 'source_file': 'climate_change.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-16T23:14:11+00:00', 'source': '..\\data\\pdf\\climate_change.pdf', 'file_path': '..\\data\\pdf\\climate_change.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-16T23:14:11+00:00', 'trapped': '', 'modDate': "D:20260516231411+00'00'", 'creationDate': "D:20260516231411+00'00'", 'page': 0, 'source_file': 'climate_change.pdf', 'file_type': 'pdf'}, page_content='Climate Change\nClimate change refers to long-term shifts in global temperatures and weather patterns. While natural\nfactors have always influenced the climate, human activities since the industrial revolution have\nbecome the main driver of change.\nThe burning of fossil fuels releases carbon dioxide and other greenhouse gases into the atmosphere,\ntrapping heat and causing global temperatures to rise.

### EMBEDDING AND VECTORSTORE DB

In [6]:
import numpy as np 
import sentence_transformers as SentenceTransformer ## Para el embbeding model
import chromadb #para el vector DB
from chromadb.config import Settings
import uuid ## todos los records en los DB deben tener un ID
from typing import List, Dict, Any, Tuple #
from sklearn.metrics.pairwise import cosine_similarity ## para el retrieval de nuestro DB


## EMBEDDING CLASS 

In [7]:
import warnings
warnings.filterwarnings("ignore") ##agregue esto porque me salen warnigns que NO quiero ver. quitar luego

In [8]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    """Se encargara de generar el embedding usando SetenceTransformer yay"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): ## metodo constructor, se ejecuta cuando se cree una instancia dentro de la clase.
        """
        Iniciar el embedding manager 
        
        Args: 
        model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name #Guarda el nombre del modelo
        self.model = None #Incia el modelo como None, vacio 
        self._load_model() #Llama al metodo que carga el modelo

        ## NOTA: el guion bajo al inicio significa METODO PRIVADO, es decir, solo se usa dentro de la clase que se creo 

    def _load_model(self): #el metodo que carga el modelo
        """Carga el modelo del Sentence Transformer"""

        try:
            print(f"Cargando el modelo: {self.model_name}")
            self.model = SentenceTransformer(self.model_name) ##descarga y carga el modelo de HuggingFace en memoria
            print(f"Se cargo el modelo con exito, yay. Dimensiones: {self.model.get_embedding_dimension()}") #tamaño del vector 
        except Exception as e: 
            print(f"fuap, no cargamo nada, mira a ver {self.model_name}: {e}")
            raise ##relanza el error para detener el programa, a diferencia de antes donde solo lo mostraba
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray: #Metodo que recibe una lista de strings y regresa un array (lista) con los embeddings 
         
         """Genera los embeddings en una lista de textos.
         
         Args: 
         
         texts: Lista de texto en un string para hacerle embedding
         nos da: Array (lista) del embedding hecho, con (len(text), embedding_dim)
         """

         if not self.model: #detiene todo si el modelo no ha sido cargado
             raise ValueError("No se cargo el modelo")
         
         print(f"Generando los embeddings para... {len(texts)} textos") #muestra la cantidad de textos que va a conventir
         embeddings = self.model.encode(texts, show_progress_bar=True) #convierte cada texto en un vector de numeros
         print(f"Se genero el embedding con forma: {embeddings.shape}") #muestra las dimensiones de los resultados en forma de (texto, cada numero x vector)
         return embeddings
    
## INICAR EL EMBEDDING MANAGER ##

embedding_manager = EmbeddingManager() #importante, los parentesis para crear la instancia
embedding_manager



Cargando el modelo: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4290.10it/s]


Se cargo el modelo con exito, yay. Dimensiones: 384


In [9]:
class VectorStore:
    """Se encargara de los embeddings dentro de ChromaDB en una vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"): #el constructor que recibira la coleccion y el directorio donde se guarda el DB
        """Inicia el vector store
        
        Args:
        collection_name: Nombre de la coleccion del ChromaDB
        persist_directory: Directorio que persiste a la vector store"""

        self.collection_name = collection_name #guarda el nombre de la coleccion como atributo
        self.persist_directory = persist_directory #guarda la ruta
        self.client = None #Inicia el cliente de ChromaDB como none, porque no hay conectado
        self.collection = None #Inicia la coleccion como none, se creara cuando inicie el Store
        self._initialize_store()

    def _initialize_store(self):
        """Inicializa el Client y collection"""
        try: 
            os.makedirs(self.persist_directory, exist_ok=True) #crea la carpeta donde se guardan los datos en caso de que no exista
            self.client = chromadb.PersistentClient(path=self.persist_directory) #crea la conexion con el ChromaDB apuntando a la carpeta creada

            self.collection = self.client.get_or_create_collection( 
                name=self.collection_name, #nombre de la collection como tabla de base de datos normal
                metadata={"descripcion": "embbedings documentos PDFs para el RAG"} #info de la collection
            )

            print(f"Se inicio la vector store. Collection: {self.collection_name}")
            print(f"Los documentos existentes en la coleccion: {self.collection.count()}") #muestra los docs guardados

        except Exception as e:
            print(f"error al iniciar la vector store: {e}")
            raise #detiene el programa completamente, sin vector store no tiene sentido continuar

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Agrega docs y sus embeddings a la vector store
        
        Args: 
            documents: Lista de langchain de documentos 
            embeddings: Los embeddings de cada documento
        """

        if len(documents) != len(embeddings): #verifica que haya un embedding por cada documento
            raise ValueError("El numero de docs debe hacer match con el numero de embeddings")
        
        ## **NOTA: Cada doc tiene el mismo numero de embedding porque un embedding representa un documento. Es "la traduccion a numero"
        ## **Su identificador, basicamente. ChromaDB guarda los docs y embedding juntos, vinculandolos x ID
        
        print(f"Agregando {len(documents)} documentos a la vector store wiwiwi...")

        ## prepara la data para la chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)): #recorre cada documento y su embedding al mismo tiempo

            ## ZIP: Pone las lineas en pares
            ## ENUMARATE: Enumera esos pares 
            ## i: El numero en enumarate. doc: los documents. embedding: los embeddings

            # 1) generar un ID unico
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" ##Genera los IDs unicos. 
            ids.append(doc_id)

            # 2) preparar metadata
            metadata = dict(doc.metadata) #convierte la metadata del documento en un diccionario editable
            metadata['doc_index'] = i #agrega el indice del documento
            metadata['content_length'] = len(doc.page_content) #agrega la longitud del contenido
            metadatas.append(metadata) #agrega la metadata a la lista

            # 3) contenido del documento 
            documents_text.append(doc.page_content) #agrega el texto del documento a la lista

            # 4) embedding 
            embeddings_list.append(embedding.tolist()) #convierte el array de numpy a lista y la agrega

        ## agregar al collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list, #los vectores numericos de cada documento
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Se agregaron de forma EXITOSA :D esta cantidad de documentos a la vector store: {len(documents)}")
            print(f"Cantidad de documentos en collection: {self.collection.count()}")

        except Exception as e:
            print(f"Mano tuvo un error poniendo los documentos en la vector store, chequee ahi {e}")
            raise

vectorstore = VectorStore()
vectorstore

Se inicio la vector store. Collection: pdf_documents
Los documentos existentes en la coleccion: 9


In [10]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-16T23:14:11+00:00', 'source': '..\\data\\pdf\\climate_change.pdf', 'file_path': '..\\data\\pdf\\climate_change.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-16T23:14:11+00:00', 'trapped': '', 'modDate': "D:20260516231411+00'00'", 'creationDate': "D:20260516231411+00'00'", 'page': 0, 'source_file': 'climate_change.pdf', 'file_type': 'pdf'}, page_content='Climate Change\nClimate change refers to long-term shifts in global temperatures and weather patterns. While natural\nfactors have always influenced the climate, human activities since the industrial revolution have\nbecome the main driver of change.\nThe burning of fossil fuels releases carbon dioxide and other greenhouse gases into the atmosphere,\ntrapping heat and causing global temperatures to rise.

In [11]:
##NOTA
# ** Este bloque va a conectar todo nuestro pipeline junto, yay

## Convertir el texto a embeddings 

texts = [doc.page_content for doc in chunks] # Extrae unicamente el raw texto de cada chunk 

## Generar los embeddings 

embeddings = embedding_manager.generate_embeddings(texts) # Toma la lista de texto y los conviertes en vectores usando el modelo anterior

## Guarda en la DB de los vectores

vectorstore.add_documents(chunks, embeddings) # Guarda todo en el ChromaDB 

Generando los embeddings para... 3 textos


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.62it/s]

Se genero el embedding con forma: (3, 384)
Agregando 3 documentos a la vector store wiwiwi...
Se agregaron de forma EXITOSA :D esta cantidad de documentos a la vector store: 3
Cantidad de documentos en collection: 12


 ### Retriever Pipeline from VectorStore ###

In [12]:
class RAGRetriever:
    """Maneja los query-based (preguntas) retrieval de la vectorstore"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager): #recone e; vectorstore y embedding como sus dependencias
        """Inicia el retriever
        
        Args: 
        vector_store: El vector store que contiene los embeddings de los documentos 
        embedding_manager: Manager para generar los embeddings de los query (preguntas)
        """

        self.vector_store = vector_store #guarda el vecto store para busquedas
        self.embedding_manager = embedding_manager #guarda el embedding manager para transformar los queries a embedding

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, any]]:
        """
        Retorna los documentos importantes para los query
        
        Args:
        query: La pregunta
        top_k: El top de resultados que nos dara
        score_threshold: El minimo de solicitud score threshold
        
        Retorna: 
            Lista de diccionarios que contiene los retrieved documents y metadata
        """     

        print(f"Buscando los documentos para la pregunta... '{query}'")
        print(f"Top k: {top_k}, score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0] #conviertr las preguntas del usuaio a un vectr numero

        try:
            resultados = self.vector_store.collection.query( #busca en los docs mas similares al vector de la preguntas
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k #cantidad max de resultados
            )

            retrieved_docs = [] # lista para los documentos encontrados

            if resultados['documents'] and resultados['documents'][0]: #verifica si hay resultados
                documents = resultados['documents'][0] #textos de los documentos encontradoss
                metadatas = resultados['metadatas'][0] #metadata de cada documento
                distances = resultados['distances'][0] # distancia entre query y cada documento
                ids = resultados['ids'][0] 

                for i, (doc_id, document, metada, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    
                    # convierte la distancia a similitud, mientras mas cercano a 1 mas similar
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold: #Filtra los docs que no superan el minimo de similitud
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metada,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1 #posicion de los docs en resultados
                        })

                print(f"Se recuperaron {len(retrieved_docs)} documentos (post-filtering)")
            else:
                print(f"No se encontraron documentos, lol")
           
            return retrieved_docs

        except Exception as e:
            print(f"Error de busqueda :( {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [13]:
rag_retriever #probar 

In [14]:
## probar con query. aqui se haran las pruebas con los multiples PDFs que jesus proporciono jiji.

rag_retriever.retrieve("" \
"")

Buscando los documentos para la pregunta... ''
Top k: 5, score threshold: 0.0
Generando los embeddings para... 1 textos


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.09it/s]

Se genero el embedding con forma: (1, 384)
Se recuperaron 0 documentos (post-filtering)


[]

### Integration VectorDB context pipeline ###

In [15]:
## Simple RAG pipeline con Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv() ## carga las variables en .env 

### iniciar el Groq LLM 

groq_api_key = os.getenv('GROQ_API_KEY')

llm = ChatGroq(api_key=groq_api_key, model_name="llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function con retrieve context y generar respuesta

def rag_simple(query, retriever, llm, top_k=3): #funcion que recibe la pregunta, retriever y llm.

    #buscar el contexto
    results = retriever.retrieve(query, top_k=top_k) #busca los elemntos mas relevantes
    context = "\n\n".join([doc['content'] for doc in results]) if results else "" #une todo el contenido en un solo texto

    if not context:
        return "No hay contexto importante para esta pregunta" 
    
    ## generar la respuesta con GROQ LLM

    prompt = f"""Con este contexto, responde la pregunta.
    
    contexto: {context}
    pregunta: {query} """ #prompt que le dice a la LLM como responder

    respuesta = llm.invoke([prompt.format(context=context, query=query)]) #envia el prompt al LLM 
    return respuesta.content #retorna solo el texto de la respuesta



In [19]:
#prueba

respuesta = rag_simple("How to improve sleep quality", rag_retriever, llm)
print(respuesta)

Buscando los documentos para la pregunta... 'How to improve sleep quality'
Top k: 3, score threshold: 0.0
Generando los embeddings para... 1 textos


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.97it/s]

Se genero el embedding con forma: (1, 384)
Se recuperaron 3 documentos (post-filtering)


Según el contexto, hay varias formas de mejorar la calidad del sueño:

1. **Mantener un horario de sueño consistente**: Esto ayuda a regular el reloj biológico y a mejorar la calidad del sueño.
2. **Limitar el tiempo de pantalla antes de acostarse**: La luz azul emitida por las pantallas puede interferir con la producción de melatonina, la hormona que regula el sueño.
3. **Mantener un ambiente de dormitorio fresco y oscuro**: Un ambiente fresco y oscuro puede ayudar a mejorar la calidad del sueño y a reducir el estrés.

Al seguir estos consejos, se puede mejorar significativamente la calidad del sueño y, por lo tanto, la salud y el bienestar en general.
